# Prompt 4 — Final LLM Explainer (Φ)

This notebook isolates the **Final Explainer prompt** used after the last optimization cycle.

The Final Explainer receives:

1. **FULL EXPERIMENTAL HISTORY**
2. **EXPLAINABILITY REPORTS FROM ALL CYCLES**
3. **CYCLE SUMMARIES**

Its task is to identify the best stable LLM-generated CMA-ES variant and reconstruct the algorithmic path that led to it, while distinguishing experimental evidence from plausible interpretation.


## 1. Final Explainer prompt and input-preparation helpers


In [ ]:
# FINAL EXPLAINER - AFTER THE LAST CYCLE

def cmaes_final_analysis_prompt(
        history: str,
        explainability_reports: str,
        cycle_summaries: str):

    return f"""
You are an expert in CMA-ES, BBOB benchmarking, evolutionary computation,
and algorithm explainability.

GOAL

Identify the best stable LLM-generated CMA-ES variant across all completed
cycles and explain which algorithmic changes most likely contributed to
its success.

INPUTS

(A) FULL EXPERIMENTAL HISTORY:
{history}

(B) EXPLAINABILITY REPORTS FROM ALL CYCLES:
{explainability_reports}

(C) CYCLE SUMMARIES:
{cycle_summaries}

TASK

Internally analyze:

1. All generated CMA-ES variants and their BBOB comparisons against
   the cycle 0 baseline.

2. Consider only generated variants that satisfy the framework's
   best-so-far eligibility rule and respect the evaluation budget.

3. Among eligible variants, identify the best variant using:

   - more IMPROVED functions first,
   - fewer WORSE functions second,
   - more SAME functions third.

4. Trace the algorithmic modifications and parent-selection path that led
   to the selected variant.

5. Determine which mechanisms were preserved, introduced, refined,
   simplified, reverted, or abandoned.

6. Explain why the successful modification appears to have worked,
   based only on the experimental evidence and recorded reports.

7. Distinguish clearly between:

   - observations directly supported by the experiments,
   - plausible mechanistic explanations,
   - and uncertain conclusions.

OUTPUT FORMAT

Return only:

BEST VARIANT:
- Cycle: ...
- Improved functions: ...
- Worse functions: ...
- Same functions: ...

KEY MODIFICATION:
- ...

WHY IT WORKED:
- ...

EVOLUTIONARY PATH:
- ...

EVIDENCE:
- ...

CONCLUSION:
- ...

RULES

- Select only among LLM-generated variants (cycles > 0) that qualified
  as stable best-so-far candidates.
- Do not select a destructive variant merely because it has a large number
  of IMPROVED functions.
- Respect the best-so-far selection criteria recorded by the framework.
- Do not claim causality when the evidence only shows association.
- Use the explainability reports to interpret why the selected mechanism
  appears effective.
- Focus on algorithmic mechanisms, not only numerical scores.
- Do not propose a new variant.
- Do not generate Python code.
- Do not use markdown tables.
"""


def cargar_reportes_explicabilidad(
        report_folder: str = "explainability_reports"):

    if not os.path.exists(report_folder):
        return "No explainability reports available."

    report_files = sorted(
        [
            file for file in os.listdir(report_folder)
            if file.startswith("explainability_report_cycle_")
            and file.endswith(".txt")
        ],
        key=lambda name: int(
            name.replace(
                "explainability_report_cycle_",
                ""
            ).replace(".txt", "")
        )
    )

    if not report_files:
        return "No explainability reports available."

    combined_reports = []

    for file_name in report_files:

        file_path = os.path.join(
            report_folder,
            file_name
        )

        with open(
            file_path,
            "r",
            encoding="utf-8"
        ) as f:

            content = f.read()

        combined_reports.append(
            f"\n===== {file_name} =====\n{content}\n"
        )

    return "\n".join(combined_reports)


def construir_resumen_ciclos_para_llm(
        cycle_summaries):

    lines = []

    for item in cycle_summaries:

        if item.get("cycle") == 0:

            lines.append(
                "Cycle 0: fixed baseline CMA-ES."
            )

            continue

        lines.append(
            (
                "Cycle {cycle}: IMPROVED={improved}, WORSE={worse}, SAME={same}, "
                "eligible_best_so_far={eligible}, became_best_so_far={became_best}, "
                "parent_cycle={parent_cycle}, parent_decision={parent_decision}"
            ).format(
                cycle=item.get("cycle"),
                improved=item.get("improved"),
                worse=item.get("worse"),
                same=item.get("same"),
                eligible=item.get("eligible_best_so_far"),
                became_best=item.get("became_best_so_far"),
                parent_cycle=item.get("parent_cycle"),
                parent_decision=item.get("parent_decision")
            )
        )

    return "\n".join(lines)


def call_llm_and_get_final_analysis(
        history: str,
        explainability_reports: str,
        cycle_summaries: str,
        model: str = "gpt-5-mini-2025-08-07"):

    print("\n--- Calling Final Explainer LLM ---")

    prompt = cmaes_final_analysis_prompt(
        history=history,
        explainability_reports=explainability_reports,
        cycle_summaries=cycle_summaries
    )

    system_prompt = """
You are an expert researcher in evolutionary computation,
CMA-ES, BBOB benchmarking, and algorithm explainability.

Identify the best stable generated CMA-ES variant from the supplied
experimental evidence and explain why its modifications appear
to have worked.

Respect the framework's best-so-far eligibility rule.
Do not generate Python code.
Do not propose a new algorithm.
Base conclusions only on the supplied history and reports.
"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]

    try:

        completion = openai.chat.completions.create(
            model=model,
            messages=messages
        )

        report = completion.choices[0].message.content

        return report.strip()

    except Exception as e:

        print(
            f"Error calling Final Explainer LLM: {e}"
        )

        return ""


## 2. Example input mapping


In [ ]:
history = """
<FULL EXPERIMENTAL HISTORY>
"""

explainability_reports = """
<EXPLAINABILITY REPORTS FROM ALL CYCLES>
"""

cycle_summaries = """
<CYCLE-BY-CYCLE SUMMARY OF IMPROVED / WORSE / SAME,
BEST-SO-FAR ELIGIBILITY, PARENT CYCLE, AND PARENT DECISION>
"""

final_explainer_prompt_text = cmaes_final_analysis_prompt(
    history=history,
    explainability_reports=explainability_reports,
    cycle_summaries=cycle_summaries
)

# print(final_explainer_prompt_text)
